## AutoCast processor training example

This notebook demonstrates training a processor directly on encoded data.

### Example dataaset

We use the `ReactionDiffusion` dataset as an example dataset to illustrate training and evaluation of models. This dataset simulates the advection-diffusion equation in 2D.


In [ ]:
from autocast.data.encoded_dataset import MiniWellDataModule, MiniWellInputOutput
from autocast.metrics.spatiotemporal import MAE, MSE, RMSE, VRMSE
from einops import rearrange, repeat
import torch
from pathlib import Path
import h5py
from torch.utils.data import ConcatDataset, Dataset, DataLoader
from autocast.types.batch import collate_encoded_samples

THE_WELL = False
simulation_name = "rayleigh_benard"
n_steps_input = 1
n_steps_output = 4
stride = 1
rollout_stride = 4

class PreprocessedInMemoryDataset(Dataset):
    def __init__(self, file_path, steps, stride, label_mean=None, label_std=None):
        with h5py.File(file_path, "r") as f:
            # Load raw data
            # shape: (Traj, T, C, H, W) e.g. (40, 200, 64, 16, 4)
            # Rearrange immediately to (Traj, T, W, H, C) for faster slicing later
            state_raw = torch.from_numpy(f["state"][:])
            self.state = rearrange(state_raw, "b t c h w -> b t w h c")
            
            # Load and normalize labels
            label_raw = torch.from_numpy(f["label"][:])
            if label_mean is not None and label_std is not None:
                label_raw = (label_raw - label_mean) / label_std
            self.label = label_raw
        
        self.trajectories = self.state.shape[0]
        self.steps_per_trajectory = self.state.shape[1]
        self.steps = steps
        self.stride = stride
        
        # Pre-calculate spatial expansion shape for labels to save time in getitem
        # self.state is (B, T, W, H, C)
        self.spatial_dims = self.state.shape[2:4] # (W, H)
        
    def __len__(self):
        return self.trajectories * (
            self.steps_per_trajectory - (self.steps - 1) * self.stride
        )

    def __getitem__(self, i):
        crops_per_trajectory = (
            self.steps_per_trajectory - (self.steps - 1) * self.stride
        )
        i, j = i // crops_per_trajectory, i % crops_per_trajectory
        
        # Slice state: (T_slice, W, H, C)
        # Slicing on dim 1 (time) which is contiguous-ish after dim 0 (traj)
        state_slice = self.state[
            i, j : j + (self.steps - 1) * self.stride + 1 : self.stride
        ]
        
        return {
            "state": state_slice,
            "label": self.label[i],
            "encoded_info": {} 
        }

class NormalizedMiniWellDataset(MiniWellInputOutput):
    def __init__(self, filepath, n_steps_input, n_steps_output, stride=1, concat_inputs_and_label=True, label_mean=None, label_std=None):
        Dataset.__init__(self)
        
        self.n_steps_input = n_steps_input
        self.n_steps_output = n_steps_output
        self.concat_inputs_and_label = concat_inputs_and_label
        
        # We pass norm stats to the sub-dataset to handle it once at load time
        l_mean = torch.tensor(label_mean) if label_mean is not None else None
        l_std = torch.tensor(label_std) if label_std is not None else None
        
        steps = n_steps_input + n_steps_output
        
        if isinstance(filepath, str):
            filepath = [filepath]
            
        print("Loading and preprocessing datasets into memory...")
        datasets = [PreprocessedInMemoryDataset(f, steps, stride, l_mean, l_std) for f in filepath]
        self.miniwell_dataset = ConcatDataset(datasets)
        print("Done loading.")

    def __len__(self):
        return len(self.miniwell_dataset)

    def __getitem__(self, index):
        data = self.miniwell_dataset.__getitem__(index)
        
        # Data is already in (T, W, H, C) format
        full_state = data["state"]
        
        input_fields = full_state[: self.n_steps_input]
        output_fields = full_state[self.n_steps_input : self.n_steps_input + self.n_steps_output]
        label = data["label"]

        if self.concat_inputs_and_label:
            # Expand label and concat
            # label: (C_label)
            # input_fields: (T_in, W, H, C_in)
            t = input_fields.shape[0]
            w, h = input_fields.shape[1], input_fields.shape[2]
            
            # Efficient implementation
            # (C) -> (1, 1, 1, C) -> expand -> (T, W, H, C)
            label_expanded = label.view(1, 1, 1, -1).expand(t, w, h, -1)
            
            # This cat is the only heavy op remaining per-sample
            input_fields = torch.cat([input_fields, label_expanded], dim=-1)

        return self.to_sample(
            {
                "input_fields": input_fields,
                "output_fields": output_fields,
                "label": label,
                "encoded_info": data.get("encoded_info", {}),
            }
        )

class CustomMiniWellDataModule(MiniWellDataModule):
    def __init__(
        self,
        dataset_cls=None,
        dataset_kwargs=None,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.dataset_cls = dataset_cls or MiniWellInputOutput
        self.dataset_kwargs = dataset_kwargs or {}

    def _create_dataset(self, dir_path):
        if not dir_path.exists():
            return None

        files = sorted(dir_path.glob("*.h5")) + sorted(dir_path.glob("*.hdf5"))
        if not files:
            return None

        common_kwargs = {
            "n_steps_input": self.n_steps_input,
            "n_steps_output": self.n_steps_output,
            "stride": self.stride,
            "concat_inputs_and_label": self.concat_inputs_and_label,
        }
        common_kwargs.update(self.dataset_kwargs)

        if len(files) == 1:
            return self.dataset_cls(filepath=str(files[0]), **common_kwargs)

        return self.dataset_cls(filepath=[str(f) for f in files], **common_kwargs)
    
    def train_dataloader(self):
        if self.train_dataset is None:
            self.setup(stage="fit")
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers, 
            collate_fn=collate_encoded_samples,
            pin_memory=True, # Faster transfer to GPU
            persistent_workers=True, # Keep workers alive
        )

base_path = (
    f"../datasets/{simulation_name}/1e3z5x2c_{simulation_name}_dcae_f32c64_large/cache/{simulation_name}"
)

# Stats for Rayleigh Benard labels
label_mean = 3.6702
label_std = 6.6332

datamodule = CustomMiniWellDataModule(
    data_path=base_path,
    n_steps_input=n_steps_input, n_steps_output=n_steps_output, stride=stride,
    batch_size=64, # Increase batch size if GPU util is still low (e.g. 128 or 256)
    num_workers=4,
    dataset_cls=NormalizedMiniWellDataset,
    dataset_kwargs={"label_mean": label_mean, "label_std": label_std}
)

### Set-up logging


In [ ]:
from autocast.logging import maybe_watch_model
from autocast.logging.wandb import create_notebook_logger

logger, watch = create_notebook_logger(
    project="autocast-notebooks",
    name=f"06_processor_{simulation_name}",
    tags=["notebook", simulation_name],
    enabled=True,
)

In [ ]:
# DIAGNOSTIC: Let's understand what's actually happening
batch = next(iter(datamodule.train_dataloader()))

print("=== Data Shapes ===")
print(f"encoded_inputs: {batch.encoded_inputs.shape}")
print(f"encoded_output_fields: {batch.encoded_output_fields.shape}")

print("\n=== Data Statistics ===")
print(f"Inputs  - mean: {batch.encoded_inputs.mean():.4f}, std: {batch.encoded_inputs.std():.4f}, min: {batch.encoded_inputs.min():.4f}, max: {batch.encoded_inputs.max():.4f}")
print(f"Outputs - mean: {batch.encoded_output_fields.mean():.4f}, std: {batch.encoded_output_fields.std():.4f}, min: {batch.encoded_output_fields.min():.4f}, max: {batch.encoded_output_fields.max():.4f}")

# Check if input and output have temporal correlation
# For a good prediction task, consecutive frames should be similar
inp = batch.encoded_inputs[:, -1, ...]  # Last input frame
out = batch.encoded_output_fields[:, 0, ...]  # First output frame
temporal_mse = ((inp[..., :out.shape[-1]] - out) ** 2).mean().item()
print(f"\nMSE between last input frame and first output frame: {temporal_mse:.4f}")

# Check variance across spatial dimensions
spatial_var = batch.encoded_output_fields.var(dim=(2, 3)).mean().item()
print(f"Mean spatial variance in outputs: {spatial_var:.4f}")

# Check if simply copying input as output would work (baseline)
# Tile input to match output time steps
baseline_pred = batch.encoded_inputs[:, -1:, ..., :batch.encoded_output_fields.shape[-1]].expand(-1, n_steps_output, -1, -1, -1)
baseline_mse = ((baseline_pred - batch.encoded_output_fields) ** 2).mean().item()
print(f"\nBaseline MSE (copy last input frame): {baseline_mse:.4f}")

# Check if predicting zeros would be better
zero_mse = (batch.encoded_output_fields ** 2).mean().item()
print(f"Baseline MSE (predict zeros): {zero_mse:.4f}")

# Check if predicting mean would be better
mean_pred = batch.encoded_output_fields.mean()
mean_mse = ((batch.encoded_output_fields - mean_pred) ** 2).mean().item()
print(f"Baseline MSE (predict global mean): {mean_mse:.4f}")

n_channels = batch.encoded_inputs.shape[-1]
w, h = batch.encoded_inputs.shape[2:4]
print(f"\nSpatial dims: {w}x{h}, Input channels: {n_channels}, Output channels: {batch.encoded_output_fields.shape[-1]}")


### Example shape and batch


In [ ]:
datamodule.train_dataset[0].encoded_inputs.shape

In [ ]:
batch = next(iter(datamodule.train_dataloader()))

batch.encoded_inputs.shape

In [ ]:

from azula.noise import VPSchedule
import torch.nn as nn

from autocast.models.processor import ProcessorModel
from autocast.nn.unet import TemporalUNetBackbone
from autocast.nn.vit import TemporalViTBackbone
from autocast.processors.flow_matching import FlowMatchingProcessor
from autocast.processors.base import Processor
from autocast.types import EncodedBatch, Tensor

batch = next(iter(datamodule.train_dataloader()))
n_channels = batch.encoded_inputs.shape[-1]
print(f"Batch Stats - Input Mean: {batch.encoded_inputs.mean():.3f}, Std: {batch.encoded_inputs.std():.3f}")
print(f"Batch Stats - Output Mean: {batch.encoded_output_fields.mean():.3f}, Std: {batch.encoded_output_fields.std():.3f}")

# processor_name = "flow_matching"  # Generative - BAD for deterministic prediction
processor_name = "direct"  # Direct regression - GOOD for deterministic prediction

n_latent_in = batch.encoded_inputs.shape[-1]
n_latent_out = batch.encoded_output_fields.shape[-1]


class DirectPredictionProcessor(Processor):
    """Direct prediction processor - predicts output directly from input (no noise/ODE)."""
    
    def __init__(
        self,
        backbone: nn.Module,
        n_steps_output: int,
        n_channels_out: int,
        loss_func: nn.Module | None = None,
        learning_rate: float = 1e-4,
        **kwargs,
    ):
        super().__init__(loss_func=loss_func or nn.MSELoss(), **kwargs)
        self.backbone = backbone
        self.n_steps_output = n_steps_output
        self.n_channels_out = n_channels_out
        self.learning_rate = learning_rate
    
    def forward(self, x: Tensor) -> Tensor:
        return self.map(x)
    
    def map(self, x: Tensor) -> Tensor:
        """Direct prediction: input -> output (no stochastic sampling)."""
        # x: (B, T_in, W, H, C_in)
        # We need to create a "dummy" time embedding since backbone expects it
        batch_size = x.shape[0]
        device, dtype = x.device, x.dtype
        
        # Use t=1 as a dummy timestep (backbone expects diffusion time)
        t = torch.ones(batch_size, device=device, dtype=dtype)
        
        # Initialize output tensor (zeros, not noise!)
        spatial_shape = tuple(x.shape[2:-1])
        z_shape = (batch_size, self.n_steps_output, *spatial_shape, self.n_channels_out)
        z = torch.zeros(z_shape, device=device, dtype=dtype)
        
        # Single forward pass through backbone
        output = self.backbone(z, t, x)
        return output
    
    def loss(self, batch: EncodedBatch) -> Tensor:
        """Simple MSE loss between prediction and target."""
        pred = self.map(batch.encoded_inputs)
        target = batch.encoded_output_fields
        return self.loss_func(pred, target)


if processor_name == "direct":
    # For direct prediction, we don't need time embedding, so use simpler backbone
    # But we can still use ViT backbone - it just ignores the time input effectively
    backbone = TemporalViTBackbone(
        in_channels=n_latent_out,
        out_channels=n_latent_out,
        cond_channels=n_latent_in,
        n_steps_output=n_steps_output,
        n_steps_input=n_steps_input,
        mod_features=256,
        hid_channels=512,
        hid_blocks=8,
        temporal_method="none",
        attention_heads=8,
        spatial=2,
        patch_size=1,
        dropout=0.05,
        ffn_factor=4,
        checkpointing=True,
    )
    
    processor = DirectPredictionProcessor(
        backbone=backbone,
        n_steps_output=n_steps_output,
        n_channels_out=n_latent_out,
        learning_rate=3e-4,
    )

elif processor_name == "flow_matching":
    backbone = TemporalViTBackbone(
        in_channels=n_latent_out,
        out_channels=n_latent_out,
        cond_channels=n_latent_in,
        n_steps_output=n_steps_output,
        n_steps_input=n_steps_input,
        mod_features=256,
        hid_channels=768,
        hid_blocks=12,
        temporal_method="attention",
        attention_heads=12,
        spatial=2,
        patch_size=1,
        dropout=0.05,
        ffn_factor=4,
        checkpointing=True,
    )
    
    processor = FlowMatchingProcessor(
        backbone=backbone,
        schedule=VPSchedule(),
        n_steps_output=n_steps_output,
        n_channels_out=n_latent_out,
        stride=stride,
        flow_ode_steps=25, 
    )
else:
    from autocast.processors.diffusion import DiffusionProcessor
    
    backbone = TemporalViTBackbone(
        in_channels=n_latent_out,
        out_channels=n_latent_out,
        cond_channels=n_latent_in,
        n_steps_output=n_steps_output,
        n_steps_input=n_steps_input,
        mod_features=256,
        hid_channels=768,
        hid_blocks=12,
        temporal_method="attention",
        attention_heads=12,
        spatial=2,
        patch_size=1,
        dropout=0.05,
        ffn_factor=4,
        checkpointing=True,
    )

    processor = DiffusionProcessor(
        backbone=backbone,
        schedule=VPSchedule(),
        n_steps_output=n_steps_output,
        n_channels_out=n_latent_out,
    )

model = ProcessorModel(
    processor=processor,
    learning_rate=3e-4,
    val_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
    test_metrics=[VRMSE(), MSE(), MAE(), RMSE()],
)
maybe_watch_model(logger, model, watch)
print(f"\nUsing processor: {processor_name}")


In [ ]:
model.val_metrics

In [ ]:
# Test: Can the model overfit a single batch?
# Key insight: Flow matching loss != MSE. We need to test properly.

import torch.optim as optim

test_batch = next(iter(datamodule.train_dataloader()))
test_batch = type(test_batch)(
    encoded_inputs=test_batch.encoded_inputs.cuda(),
    encoded_output_fields=test_batch.encoded_output_fields.cuda(),
    label=test_batch.label,
    encoded_info=test_batch.encoded_info,
)

# Baseline: what MSE do we get by just copying the last input frame?
baseline_pred = test_batch.encoded_inputs[:, -1:, ..., :test_batch.encoded_output_fields.shape[-1]].expand(-1, n_steps_output, -1, -1, -1)
baseline_mse = ((baseline_pred - test_batch.encoded_output_fields) ** 2).mean().item()
print(f"Baseline MSE (copy input): {baseline_mse:.4f}")

# CRITICAL TEST: Try a simple direct regression model instead of flow matching
# This will tell us if the TASK is learnable
class SimpleDirectPredictor(torch.nn.Module):
    def __init__(self, in_ch, out_ch, t_in, t_out, w, h):
        super().__init__()
        self.t_out = t_out
        self.out_ch = out_ch
        self.w = w
        self.h = h
        # Simple MLP that directly predicts output from input
        in_flat = t_in * w * h * in_ch
        out_flat = t_out * w * h * out_ch
        self.net = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(in_flat, 2048),
            torch.nn.ReLU(),
            torch.nn.Linear(2048, 2048),
            torch.nn.ReLU(),
            torch.nn.Linear(2048, out_flat),
        )
    
    def forward(self, x):
        out = self.net(x)
        return out.view(-1, self.t_out, self.w, self.h, self.out_ch)

simple_model = SimpleDirectPredictor(
    in_ch=test_batch.encoded_inputs.shape[-1],
    out_ch=test_batch.encoded_output_fields.shape[-1],
    t_in=n_steps_input,
    t_out=n_steps_output,
    w=test_batch.encoded_inputs.shape[2],
    h=test_batch.encoded_inputs.shape[3],
).cuda()

optimizer_simple = optim.Adam(simple_model.parameters(), lr=1e-3)
print("\nTesting SIMPLE direct prediction model (100 steps)...")
for i in range(100):
    optimizer_simple.zero_grad()
    pred = simple_model(test_batch.encoded_inputs)
    loss = ((pred - test_batch.encoded_output_fields) ** 2).mean()
    loss.backward()
    optimizer_simple.step()
    if i % 20 == 0:
        print(f"Step {i}: MSE={loss.item():.4f}")

print(f"\nSimple model can achieve MSE: {loss.item():.4f}")
print(f"This proves the TASK is learnable. If flow matching can't match this, the issue is with the generative approach.\n")

# Now test the flow matching model with MORE ODE steps
print("="*50)
print("Testing Flow Matching with different ODE step counts...")

# Temporarily increase ODE steps and test
model_test = model.cuda()
optimizer = optim.Adam(model_test.parameters(), lr=1e-3)

# Train for more steps
print("\nTraining flow matching for 100 steps...")
for i in range(100):
    optimizer.zero_grad()
    loss = model_test.processor.loss(test_batch)
    loss.backward()
    optimizer.step()
    if i % 25 == 0:
        # Test with different ODE step counts
        # original_steps = model_test.processor.flow_ode_steps
        for ode_steps in [25, 50, 100]:
            # model_test.processor.flow_ode_steps = ode_steps
            with torch.no_grad():
                pred = model_test.processor.map(test_batch.encoded_inputs)
                mse = ((pred - test_batch.encoded_output_fields) ** 2).mean()
            print(f"Step {i}, ODE steps={ode_steps}: loss={loss.item():.4f}, MSE={mse.item():.4f}")
        # model_test.processor.flow_ode_steps = original_steps
        print()


### Run trainer


In [ ]:
import lightning as L

# device = "mps"  # "cpu"
device = "cuda"  # "cpu"
# Removed limit_train_batches=0.1 to use full data
trainer = L.Trainer(
    max_epochs=10, accelerator=device, logger=logger,
    limit_train_batches=1.0, limit_val_batches=0.1
)
trainer.fit(model, datamodule)
trainer.save_checkpoint(f"./{simulation_name}_{processor_name}_model.ckpt")


### Run the evaluation


In [ ]:
trainer.test(model, datamodule)